In [1]:
from PIL import Image
import win32com.client
import photoshop.api as ps
import photoshop
import time
import pandas as pd
import datetime
import requests
from lxml import etree
import requests
import os
import re
import subprocess
#处理图片，切掉图片里面的透明部分
def save_image():
    ps_app = win32com.client.Dispatch("Photoshop.Application")
    # 获取当前活动文档
    doc = ps_app.ActiveDocument
    # 获取文档中的所有图层
    layers = doc.ArtLayers
    # 从后向前遍历并删除除了第一个图层外的所有其他图层
    for i in range(layers.Count - 1, 0, -1):
        layer = layers[i]
        layer.Delete()
    # 创建Photoshop应用程序对象
    # 获取当前活动文档
    doc = ps_app.ActiveDocument
    # 设置PNG格式选项
    png_options = win32com.client.Dispatch("Photoshop.ExportOptionsSaveForWeb")
    png_options.Format = 13  # 使用PNG格式
    png_options.PNG8 = False  # 使用PNG-24格式
    png_options.Transparency = True  # 启用透明度
    # 导出文档为PNG图片
    save_path = "C:\\Users\\86139\\Desktop\\猪猪数据备份\\output.png"
    doc.Export(ExportIn=save_path, ExportAs=2, Options=png_options)
    print("图片保存成功")
    # 打开PNG图片
    img = Image.open(save_path)
    # 裁剪掉透明部分
    img = img.crop(img.getbbox())
    # 保存裁剪后的图片
    img.save(save_path)
#将图片全部前移
def com_image():
    print('调整图片的位置')
     # 创建Photoshop COM对象
    ps_app = win32com.client.Dispatch("Photoshop.Application")
    # 读取JavaScript脚本文件
    script_file_path ="C:\\Users\\86139\\Desktop\\猪猪数据备份\\调整文字图层的位置.jsx"
    with open(script_file_path,"r",encoding="utf-8") as file:
        javascript_code = file.read()
    # 执行JavaScript脚本
    ps_app.DoJavaScript(javascript_code)
    print('正在合并图片')
    app = ps.Application()
    doc = app.activeDocument
    for i in range(5):
        j = 5-i
        k = 4-i
        first_layer = doc.layers[k]
        second_layer = doc.layers[j]
        first_layer.merge()
        # 确保合并后的图层被选中
        merged_layer = doc.activeLayer
        merged_layer.selected = True
def copy_image():
    psApp = win32com.client.Dispatch("Photoshop.Application")
    psDoc = psApp.ActiveDocument
    # 循环复制1-6号图层
    s = 0
    layer_total = []
    for i in range(1,7):
        i = i+s
        original_layer = psDoc.ArtLayers[i]
        layer = original_layer.Duplicate()
        layer_total.append(layer)
        s = s+1
    psDoc = psApp.ActiveDocument
    # 创建两个空白图层
    new_layer1 = psDoc.ArtLayers.Add()
    new_layer1.Name = "1-30"  # 设置第一个新图层的名称
    new_layer2 = psDoc.ArtLayers.Add()
    new_layer2.Name = "Blank Layer 2"  # 设置第二个新图层的名称
    # 将第二个图层移动到第一个图层上面，并合并
    psDoc.ActiveLayer = new_layer2  # 设置第一个图层为活动图层
    new_layer2.Merge()  # 合并第一个图层和第二个图层
def handle_content(content,rank):
    print("正在更换事件和热度")
    content = "\r".join(content.split("\n"))
    content = content.lstrip("\r")
    rank = "\r".join(rank.split("\n"))
    rank = rank.lstrip("\r")
    psApp = win32com.client.Dispatch("Photoshop.Application")
    psDoc = psApp.ActiveDocument
    # 获取第四个图层（假设是文字图层）# 更改图层中的文字内容
    text_layer = psDoc.ArtLayers.Item(4) 
    text_layer.TextItem.Contents = content
    rank_layer = psDoc.ArtLayers.Item(5)  
    rank_layer.TextItem.Contents = rank
    #修改字距
    ps_app = win32com.client.Dispatch("Photoshop.Application")
    script_file_path ="C:\\Users\\86139\\Desktop\\猪猪数据备份\\修改字距.jsx"
    with open(script_file_path,"r",encoding="utf-8") as file:
        javascript_code = file.read()
    ps_app.DoJavaScript(javascript_code)
def open_psd():
    print("正在打开ps，并等待25秒")
    # 连接到Photoshop
    app = win32com.client.Dispatch("Photoshop.Application")
    # 打开PSD文件
    psd_file_path = r"C:\Users\86139\Desktop\猪猪数据备份\文字内容.psd"
    doc = app.Open(psd_file_path)
    time.sleep(25)
def get_content():
    print("正在读取事件和热度")
    time_stamp = datetime.datetime.now()
    time_now = time_stamp.strftime('%Y.%m.%d')
    path = "C:\\Users\\86139\\Desktop\\保存热搜数据\\"+time_now+"微博热搜的数据.xlsx"
    data = pd.read_excel(path)
    data = data.dropna()
    df_content = ['事件']
    df_rank = ['热度']
    for i in range(21):
        df_content.append(data.iloc[i,1])
        if i == 0:
            df_rank.append(' ')
        else:
            df_rank.append(str(int(data.iloc[i,2])))
    content = "\n".join(df_content)
    rank = "\n".join(df_rank)
    print(content)
    return content,rank
def quit_ps():
    print("正在等待5s")
    time.sleep(5)
    print("ps操作完成，关闭ps")
    ps_app = win32com.client.Dispatch("Photoshop.Application")
    doc = ps_app.ActiveDocument
    # 关闭文档并选择不保存
    doc.Close(2)  # 参数2表示不保存
    ps_app.Quit()
def create_dateframe(html):
    digtal = re.compile('[0-9]+')
    text_list = []
    tr_list = html.xpath('//*[@id="pl_top_realtimehot"]/table/tbody/tr')
    data = pd.DataFrame(columns = ['排行','事件','热度','链接','事件拼接链接'])
    for tr in tr_list:
        tr_rank = tr.xpath('./td[1]/text()')
        if tr_rank == []:
            tr_rank = '上升'
            tr_text = tr.xpath('./td[2]/a/text()')[0]
            tr_heat = ''
            tr_href = 'https://s.weibo.com' +tr.xpath('./td[2]/a/@href')[0]
            tr_splice = tr_rank+'.'+ tr_text +' '+ tr_href
        else:
            tr_rank = tr_rank[0]
            tr_text = tr.xpath('./td[2]/a/text()')[0]
            tr_heat = tr.xpath('./td[2]/span/text()')[0]
            tr_heat = digtal.findall(tr_heat)
            tr_splice = tr_rank+'.'+ tr_text+' '+ tr_href
            if tr_heat != []:
                tr_heat = tr_heat[0]
            else:
                tr_heat = ''
            tr_href = 'https://s.weibo.com' + tr.xpath('./td[2]/a/@href')[0]
        data = pd.concat([data,pd.DataFrame([tr_rank,tr_text,tr_heat,tr_href,tr_splice],index = ['排行','事件','热度','链接','事件拼接链接']).T])
        data = data.reset_index(drop=True)
    return data
def save_excel(df,time_stamp):
    time_now = time_stamp.strftime('%Y.%m.%d')
    os.chdir("C:\\Users\\86139\\Desktop\\保存热搜数据")
    file_name = time_now+'微博热搜的数据.xlsx'
    df.to_excel(file_name,index = False)
    print(time_now,'数据更新成功')
def paqushuju():
    print("正在爬取微博热搜数据")
    url = 'https://s.weibo.com/top/summary'
    headers = {
        'user-agent': 'Mozilla/5.0 (Windows NT 10.0; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/104.0.0.0 Safari/537.36',
        'referer': 'https://passport.weibo.com/',
        'cookie': 'SUB=_2AkMS-WPBf8NxqwFRmfoRxGzibo9zyQvEieKkpZIaJRMxHRl-yT9kqkxdtRB6OXlNLnasnhc3GMLU1lxeZgVSuqQcvtNF; SUBP=0033WrSXqPxfM72-Ws9jqgMF55529P9D9WW.zgb4bVoLSbXXhE74YvNV; _s_tentry=passport.weibo.com; Apache=4906935630120.408.1705372918261; SINAGLOBAL=4906935630120.408.1705372918261; ULV=1705372918271:1:1:1:4906935630120.408.1705372918261:'
    }
    time_stamp = datetime.datetime.now()
    response = requests.get(url = url,headers = headers)
    response.raise_for_status()
    response.encoding = response.apparent_encoding
    html_1 = etree.HTML(response.text)
    data = create_dateframe(html_1)
    save_excel(data,time_stamp)
def handle_AE():
    print("正在执行AE的操作")
    # 定义After Effects可执行文件的路径
    after_effects_path = r"D:\应用\AE\数据存放\Adobe After Effects 2022\Support Files\AfterFX.exe"
    # 指定要打开的项目文件路径
    project_file_path = r"C:\Users\86139\Desktop\猪猪数据备份\猪猪的动态 副本.aep"
    # 使用subprocess模块调用After Effects并打开指定的项目文件
    process1 = subprocess.Popen([after_effects_path, project_file_path],shell = True)
    print("等待30s打开AE及项目")
    time.sleep(30)
    print('打开项目完成')
    # 执行对应的jsx代码
    script_path = r"C:\Users\86139\Desktop\猪猪数据备份\测试脚本.jsx" # 将路径替换为你的jsx脚本的路径
    after_effects_path = r"D:\应用\AE\数据存放\Adobe After Effects 2022\Support Files\AfterFX.exe"  # 替换为你的AE安装路径
    process2 = subprocess.Popen([after_effects_path, "-r", script_path])
    print("等待25s，完成视频输出。")
    time.sleep(25)
    print('视频输出完成')
    process2.terminate()

In [2]:
#运行代码
paqushuju()
#ps操作
open_psd()
content,rank = get_content()
handle_content(content,rank)
copy_image()
com_image()
save_image()
quit_ps()
#AE操作
handle_AE()

正在爬取微博热搜数据
2024.02.17 数据更新成功
正在打开ps，并等待25秒
正在读取事件和热度
事件
傅园慧长白山发博求助
iPhone16或垂直排列摄像头
春节返程让人泪目的离别瞬间
贾玲高马尾帅上了另一个level
吉林文旅回应傅园慧求助
王鹤棣破中国明星名人赛历史记录
张艺兴女团正面照
日本男子被同事推入15米深山谷
沈月一个人就是一个团队
红毯先生上映7天票房仅8000万元
白敬亭你好星期六发带剧照
机票太贵了回不了学校
吉林文旅
撤档
切尔诺贝利的狼已进化出抗癌能力
王一博与穿山甲合照
种地吧
孙龙
毕志飞想请薛之谦出演电影屏摄
因为不想上班在返程高铁上流泪
男子记错开工时间提前1天上班
正在更换事件和热度
调整图片的位置
正在合并图片
图片保存成功
正在等待5s
ps操作完成，关闭ps
正在执行AE的操作
等待30s打开AE及项目
打开项目完成
等待25s，完成视频输出。
视频输出完成


In [3]:
import webbrowser
webbrowser.register('360se', None, webbrowser.BackgroundBrowser("D:\\应用\\360浏览器\\360se6\\Application\\360se.exe"))
webbrowser.get('360se').open("https://member.bilibili.com/platform/upload/video/frame?spm_id_from=333.1007.top_bar.upload")


True